In [2]:
import sys
import logging
from elasticsearch import Elasticsearch
from elasticsearch.helpers import scan
from tqdm import tqdm
import pandas as pd
from tqdm import tqdm
from airflow.models import Variable

url = 'https://linode-es.trademo.com:9600'
ELASTIC_RPS_USERNAME = 'dev_user'
ELASTIC_RPS_PASSWORD = 'WW2k28HHcg'

es = Elasticsearch([url], http_auth=(ELASTIC_RPS_USERNAME, ELASTIC_RPS_PASSWORD),timeout = 30, verify_certs = False)


/home/ubuntu/common-3.11-env/lib/python3.11/site-packages/airflow/configuration.py:761 FutureWarning: The auth_backends setting in [api] has had airflow.api.auth.backend.session added in the running config, which is needed by the UI. Please update your config before Apache Airflow 3.0.

/tmp/ipykernel_741167/973597857.py:14 DeprecationWarning: The 'timeout' parameter is deprecated in favor of 'request_timeout'

/home/ubuntu/common-3.11-env/lib/python3.11/site-packages/elasticsearch/_sync/client/__init__.py:403 SecurityWarning: Connecting to 'https://linode-es.trademo.com:9600' using TLS with verify_certs=False is insecure

/tmp/ipykernel_741167/973597857.py:14 DeprecationWarning: The 'http_auth' parameter is deprecated. Use 'basic_auth' or 'bearer_auth' parameters instead

In [3]:
def fetch_logs_by_la(la, date_type , start_date, end_date, index_name = 'rps_ingestion_logs'):
    query = {
        'track_total_hits':True,
        'query': {
            'bool': {
                'filter': [
                    {'term': {'list_abbreviation': la}},
                    {"range": {
                        date_type: {
                            "gte": start_date,
                            "lte": end_date
                        }
                    }}
                ],
                'must_not':[
                    {'term': {'deleted': True}}
                ]
            }
        }
        # '_source' : req_fields
    }

    
    # Execute the search using the scan helper
    response = scan(client=es, index= index_name, query=query)
    data = [doc for doc in response]
    res_df = pd.DataFrame(data)
    # df = pd.DataFrame(list(res_df))
    # df['es_id'] = res_df['_id']
    return res_df

In [4]:
logs = fetch_logs_by_la('GG-GFSC-BB', 'logged_at' , '2026-01-29', '2026-01-29', index_name = 'rps_ingestion_logs')

logs_data = pd.DataFrame(list(logs['_source']))

logs_data['data'] = logs_data['data'].apply(eval)

logs_data_ = pd.DataFrame(list(logs_data.sort_values(by = 'logged_at', ascending = False)['data']))[['modified_cols_name','record_delta_v1', 'updated_in_db_date', 'unique_key']]


[2026-01-29T07:43:52.219+0000] {_transport.py:372} INFO - POST https://linode-es.trademo.com:9600/rps_ingestion_logs/_search?scroll=5m [status:N/A duration:129.335s]
[2026-01-29T07:43:52.220+0000] {_node_pool.py:249} WARNING - Node <Urllib3HttpNode(https://linode-es.trademo.com:9600)> has failed for 1 times in a row, putting on 1 second timeout


ConnectionTimeout: Connection timed out

In [ ]:
print('hi')